# Video 8: Multi-Agent Collaboration

## What We Will Build
In this notebook, we will build a **Research, Analyst, and Writer Crew** that demonstrates:
- Agent delegation and collaboration
- Task handoffs between specialized agents
- Shared context across agents
- Manager-agent pattern
- Multi-agent orchestration
- Review and feedback loops

By the end, you will have a complete multi-agent system that produces a final report.

## 1. Install & Import

In [ ]:
!pip install -q crewai crewai-tools langchain-openai python-dotenv

## 2. Imports & Setup

In [ ]:
from crewai import Agent, Task, Crew, Processfrom langchain_openai import ChatOpenAIimport osfrom dotenv import load_dotenvload_dotenv()llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.7, api_key=os.getenv('OPENAI_API_KEY'))print('Setup complete')

## 3. Theory: Multi-Agent Collaboration

### Why Multiple Agents?
- **Specialization**: Each agent excels in one domain
- **Quality**: Multiple perspectives improve output
- **Scale**: Complex tasks can be parallelized
- **Review**: Agents can review each other's work

### Collaboration Patterns
| Pattern | Description | Use Case |
|---------|-------------|----------|
| **Sequential** | Handoffs in a pipeline | Content creation |
| **Iterative** | Review and revise cycles | Quality assurance |
| **Parallel** | Agents work simultaneously | Research |
| **Hierarchical** | Manager coordinates | Complex projects |

### Task Handoffs
When Agent A completes a task, its output becomes context for Agent B. This creates a natural handoff.

## 4. Create Specialized Agents with Delegation

In [ ]:
# Research Agent - gathers informationresearcher = Agent(    role='Lead Research Specialist',    goal='Conduct thorough research and provide comprehensive findings',    backstory=(        'You are the lead researcher on the team. You dig deep into topics, find reliable sources, '        'and present findings in a structured format. You are thorough and detail-oriented.'    ),    llm=llm,    verbose=True,    allow_delegation=True,  # Can delegate sub-tasks    max_iter=5)# Data Analyst Agent - processes and analyzesdata_analyst = Agent(    role='Senior Data Analyst',    goal='Extract insights, identify trends, and provide data-driven recommendations',    backstory=(        'You are a data analyst who transforms raw research into actionable insights. '        'You identify patterns others miss and provide strategic recommendations.'    ),    llm=llm,    verbose=True,    allow_delegation=True,    max_iter=5)# Writer Agent - creates contentwriter = Agent(    role='Senior Content Writer',    goal='Create compelling, well-structured reports and content',    backstory=(        'You are a professional writer who crafts engaging narratives from complex information. '        'Your reports are clear, concise, and impactful.'    ),    llm=llm,    verbose=True,    allow_delegation=True,    max_iter=5)# Review Agent - quality assurancereviewer = Agent(    role='Content Reviewer',    goal='Ensure all content is accurate, complete, and well-structured',    backstory=(        'You are a meticulous reviewer who ensures quality. You check for accuracy, completeness, '        'clarity, and structure. You provide constructive feedback for improvement.'    ),    llm=llm,    verbose=True,    allow_delegation=False,    max_iter=3)print('4 specialized agents created with delegation enabled')

## 5. Define Tasks with Dependencies

In [ ]:
# Task 1: Researchresearch_task = Task(    description=(        'Research the current state of electric vehicles (EVs) in 2025. Cover: '        '1) Market size and growth, 2) Major manufacturers and models, '        '3) Battery technology advances, 4) Charging infrastructure, '        '5) Government policies and incentives. Provide specific data points.'    ),    expected_output=(        'A comprehensive research report with specific data points, statistics, and examples. '        'Minimum 500 words with clear sections.'    ),    agent=researcher)# Task 2: Analysis (depends on research)analysis_task = Task(    description=(        'Analyze the EV research findings. Identify: 1) Key market trends, '        '2) Competitive landscape insights, 3) Technology trajectory, '        '4) Investment opportunities, 5) Risks and challenges. '        'Provide strategic recommendations.'    ),    expected_output=(        'A strategic analysis with trends, insights, and actionable recommendations. '        'Minimum 400 words.'    ),    agent=data_analyst,    context=[research_task])# Task 3: Writing (depends on research and analysis)writing_task = Task(    description=(        'Write a comprehensive industry report on Electric Vehicles in 2025. '        'Use the research and analysis provided. Include: executive summary, '        'market overview, technology analysis, competitive landscape, '        'future outlook, and recommendations. Make it professional and engaging.'    ),    expected_output=(        'A polished industry report of at least 600 words with professional formatting '        'and clear sections.'    ),    agent=writer,    context=[research_task, analysis_task])# Task 4: Review (depends on writing)review_task = Task(    description=(        'Review the EV industry report. Check for: 1) Factual accuracy, '        '2) Completeness of sections, 3) Clarity and readability, '        '4) Structure and flow, 5) Actionability of recommendations. '        'Provide a review summary with a quality score (1-10) and specific feedback.'    ),    expected_output=(        'A detailed review with quality score, strengths, weaknesses, and improvement suggestions.'    ),    agent=reviewer,    context=[writing_task])print('Tasks defined with dependencies:')print('  Research -> Analysis -> Writing -> Review')

## 6. Build the Collaboration Crew

In [ ]:
collaboration_crew = Crew(    agents=[researcher, data_analyst, writer, reviewer],    tasks=[research_task, analysis_task, writing_task, review_task],    process=Process.sequential,    verbose=True,    memory=True)print('Multi-agent collaboration crew built')print(f'  Agents: {len(collaboration_crew.agents)}')print(f'  Tasks: {len(collaboration_crew.tasks)}')print(f'  Process: {collaboration_crew.process}')

## 7. Execute the Collaboration Workflow

In [ ]:
print('\n' + '=' * 70)print('EXECUTING MULTI-AGENT COLLABORATION WORKFLOW')print('=' * 70)result = collaboration_crew.kickoff()print('\n' + '=' * 70)print('FINAL OUTPUT')print('=' * 70)print(result)

## 8. Inspect Individual Agent Outputs

In [ ]:
print('\n' + '=' * 70)print('INDIVIDUAL TASK OUTPUTS')print('=' * 70)tasks = [research_task, analysis_task, writing_task, review_task]for task in tasks:    print(f'\n--- {task.agent.role}: {task.description[:50]}... ---')    if hasattr(task, 'output') and task.output:        print(task.output.raw[:300] + '...' if len(task.output.raw) > 300 else task.output.raw)    else:        print('Output not available')

## 9. Collaboration Flow Visualization

```
Researcher              Data Analyst              Writer              Reviewer
    |                       |                       |                    |
    ▼                       ▼                       ▼                    ▼
Research Task  ──────>  Analysis Task  ──────>  Writing Task  ──────>  Review Task
    │                       │                       │                    │
    │  Raw findings         │  Insights             │  Draft report      │  Feedback
    │                       │                       │                    │
    └───────────────────────┴───────────────────────┴────────────────────┘
                                    │
                                    ▼
                              Final Report
```

## 10. Key Collaboration Principles

1. **Clear roles**: Each agent knows its specialty
2. **Explicit dependencies**: Use `context` to pass outputs
3. **Allow delegation**: Enable `allow_delegation` for flexibility
4. **Review loops**: Add review agents for quality
5. **Memory**: Enable for context continuity
6. **Verbose logging**: Track agent decisions and handoffs

---

## ✅ Summary

You have successfully:
1. ✅ Created 4 specialized agents (Researcher, Analyst, Writer, Reviewer)
2. ✅ Defined tasks with clear dependencies
3. ✅ Enabled agent delegation for collaboration
4. ✅ Executed a sequential collaboration workflow
5. ✅ Inspected individual agent outputs
6. ✅ Observed task handoffs and context passing

In the next notebook, we will explore **CrewAI Flows and Advanced Workflows**.